<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator — Google Colab（LM Link）

ColabでLM Studio（llmster）を動かし、LM Link経由でPCと接続するのじゃ。認証情報は`dbv4-wd14-tagger`と共通のGoogleドライブフォルダーへ保存する。モデルはpfnet公式のGGUFファイルだけを使用し、既定は8B／Q4_K_Mじゃ。

## 1. 共通設定

Googleドライブの保存先、LM Studioの再ログイン、PLaMoのモデルサイズ・量子化形式・同時推論数を先にまとめて選択するのじゃ。

In [ ]:
#@title ⚙️ すべての設定
#@markdown **Googleドライブ**（dbv4-wd14-taggerと共通の保存先）

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = ".colab-secret" #@param {type:"string"}

#@markdown **LM Studioの認証**
FORCE_RELOGIN = False #@param {type:"boolean"}

#@markdown **PLaMo 3 Translateのモデル**（同時ロードは1つのみ）
plamo_size = "8b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0"]
PARALLEL_REQUESTS = 4 # @param {type:"integer"}

if not 1 <= PARALLEL_REQUESTS <= 16:
    raise ValueError("同時推論数は1から16で指定するのじゃ")

CONFIG_DRIVE_FOLDER = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else ".colab-secret"
if (
    CONFIG_DRIVE_FOLDER in (".", "..")
    or "/" in CONFIG_DRIVE_FOLDER
    or "\\" in CONFIG_DRIVE_FOLDER
):
    raise ValueError("Googleドライブの保存先は単一フォルダー名を指定するのじゃ")
print(
    "[INFO] Googleドライブ保存: "
    + (f"有効 (MyDrive/{CONFIG_DRIVE_FOLDER})" if USE_GOOGLE_DRIVE else "無効")
)

## 2. GPU別のPLaMo実行目安

**Colabが割り当てるGPUは毎回変わる**ため、次のセルでGPU名と現在のVRAM総量・使用量・空き容量を実測する。

| GPU（公称VRAM） | 推奨するGGUF | 31Bの目安 |
|---|---|---|
| T4（16GB） | 0.8B / 2B / 8B：Q4_K_M | 全レイヤーGPUロードは推奨しない |
| L4（24GB） | 0.8B / 2B / 8B：Q4_K_M | Q2_K / Q3_K_Mは候補。Q4_K_Mは余裕が小さく要検証 |
| A100（40GB） | 全サイズ：Q4_K_M | Q4_K_Mを候補にできる |
| A100（80GB）・H100（80GB） | 全サイズ：Q4_K_M | Q4_K_Mを候補にできる |

**PLaMo公式GGUFのファイル容量（十進GB）**

| サイズ | Q4_K_M | Q4_0 | Q3_K_M | Q2_K |
|---|---:|---:|---:|---:|
| 0.8B | 0.57 | 0.55 | 0.49 | 0.41 |
| 2B | 1.81 | 1.70 | 1.50 | 1.23 |
| 8B | 5.28 | ― | 4.31 | 3.45 |
| 31B | 19.53 | ― | 15.86 | 12.61 |

**判定方法：** 下のセルは空きVRAMとファイル容量を比較し、4GiB以上の余裕があれば「容量上の候補」、本体だけ収まれば「余裕小」、本体も収まらなければ「GPU全量は困難」と表示する。この4GiBは便宜的な目安で、**実測の必要VRAMではない**。実際のGPU使用量はLM Studioのコンテキスト長・並列数・KVキャッシュ・GPUオフロードによって変わる。CPUオフロードで動く場合もある。**候補表示はロード成功や4並列動作の保証ではない**。

公式資料：[0.8B](https://huggingface.co/pfnet/plamo-3-translate-0.8b-GGUF) / [2B](https://huggingface.co/pfnet/plamo-3-translate-2b-GGUF) / [8B](https://huggingface.co/pfnet/plamo-3-translate-8b-GGUF) / [31B](https://huggingface.co/pfnet/plamo-3-translate-31b-GGUF)、[NVIDIA GPU仕様](https://www.nvidia.com/ja-jp/data-center/graphics-cards-for-virtualization/)、[ColabのGPU割当について](https://research.google.com/colaboratory/faq.html?hl=ja)。


In [ ]:
#@title GPU・VRAMの検出とモデル容量の目安
import csv
import io
import subprocess

PLAMO_GGUF_SIZES_GB = {
    "0.8b": {"Q4_K_M": 0.57, "Q4_0": 0.55, "Q3_K_M": 0.49, "Q2_K": 0.41},
    "2b": {"Q4_K_M": 1.81, "Q4_0": 1.70, "Q3_K_M": 1.50, "Q2_K": 1.23},
    "8b": {"Q4_K_M": 5.28, "Q3_K_M": 4.31, "Q2_K": 3.45},
    "31b": {"Q4_K_M": 19.53, "Q3_K_M": 15.86, "Q2_K": 12.61},
}
GPU_VRAM_MARGIN_GIB = 4.0

try:
    gpu_output = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=15, encoding="utf-8"
    ).stdout
except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
    print(f"[WARN] NVIDIA GPU情報を取得できぬ: {error}", flush=True)
    gpu_output = ""

gpu_rows = list(csv.reader(io.StringIO(gpu_output), skipinitialspace=True))
if not gpu_rows:
    print("[WARN] NVIDIA GPUを検出できぬ。ランタイムのGPU設定を確認するのじゃ。", flush=True)
for row in gpu_rows:
    if len(row) != 5:
        print(f"[WARN] nvidia-smiの出力形式が想定外じゃ: {row}", flush=True)
        continue
    try:
        gpu_index, gpu_name = row[0].strip(), row[1].strip()
        total_mib, used_mib, free_mib = [int(v.strip()) for v in row[2:]]
    except ValueError as error:
        print(f"[WARN] GPUのVRAMを解析できぬ: {error}", flush=True)
        continue
    free_gib = free_mib / 1024
    print(
        f"GPU {gpu_index}: {gpu_name} | 総VRAM {total_mib / 1024:.2f} GiB | "
        f"使用 {used_mib / 1024:.2f} GiB | 空き {free_gib:.2f} GiB",
        flush=True
    )
    print(" PLaMo / 量子化       GGUF容量    GPU全量ロードの容量目安", flush=True)
    for size, variants in PLAMO_GGUF_SIZES_GB.items():
        for quant, gb in variants.items():
            weight_gib = gb * (10 ** 9) / (1024 ** 3)
            if free_gib >= weight_gib + GPU_VRAM_MARGIN_GIB:
                status = "容量上の候補（実行保証なし）"
            elif free_gib >= weight_gib:
                status = "余裕小・要検証"
            else:
                status = "GPU全量は困難（CPUオフロードは別）"
            chosen = " ← 選択中" if size == plamo_size and quant == quantization else ""
            print(f" {size:>4} / {quant:<7}      {gb:>5.2f} GB    {status}{chosen}", flush=True)
print("[INFO] 4GiBの余裕は便宜的な比較基準じゃ。実VRAM必要量はロード時に確認するのじゃ。", flush=True)


## 3. LM Studioのインストール

先にインストールし、次のセルで認証情報を復元してからデーモンを起動する。

In [ ]:
#@title LM Studioのインストール
import codecs
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

def run_console(command, check=True):
    if shutil.which("script") is None:
        raise RuntimeError("PTY用のscriptコマンドが見つからぬ。util-linuxを確認するのじゃ")
    argv = [str(item) for item in command]
    print("\n$ " + shlex.join(argv), flush=True)
    process = subprocess.Popen(
        ["script", "-q", "-f", "-e", "-c", shlex.join(argv), "/dev/null"],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=0
    )
    decoder = codecs.getincrementaldecoder("utf-8")("replace")
    chunks = []
    try:
        while True:
            data = process.stdout.read(8192)
            if not data:
                break
            decoded = decoder.decode(data)
            chunks.append(decoded)
            sys.stdout.write(decoded)
            sys.stdout.flush()
        decoded = decoder.decode(b"", final=True)
        chunks.append(decoded)
        sys.stdout.write(decoded)
        sys.stdout.flush()
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
        process.stdin.close()
    print(f"[EXIT] {code}: {shlex.join(argv)}", flush=True)
    if check and code:
        raise subprocess.CalledProcessError(code, argv)
    return subprocess.CompletedProcess(argv, code, stdout="".join(chunks))

run_console(["pkill", "-x", "llmster"], check=False)
run_console(["pkill", "-x", "lmstudio"], check=False)
run_console(["bash", "-o", "pipefail", "-c", "curl -fL https://lmstudio.ai/install.sh | bash"])
os.environ["PATH"] = str(Path.home() / ".lmstudio" / "bin") + os.pathsep + os.environ["PATH"]
run_console(["lms", "bootstrap"])
run_console(["sync"])
print("LM Studioのセットアップが完了したのじゃ。")


## 4. Googleドライブへのログイン状態保存

DBV4のTailscaleと同様、LM Studioのログイン状態をGoogleドライブの `MyDrive/.colab-secret/lmstudio/` 以下へ保存・自動復元する。パスフレーズの入力やColabシークレットは不要じゃ。保存対象は `credentials` と `hub` の認証状態のみで、内部デーモンキーやモデル本体は保存しない。

In [ ]:
#@title 認証情報の復元と保存
import os
import shutil
from pathlib import Path

from google.colab import drive

LM_HOME = Path.home() / ".lmstudio"
DRIVE_AUTH_DIR = None
AUTH_DIRS = ("credentials", "hub")

if USE_GOOGLE_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_AUTH_DIR = Path("/content/drive/MyDrive") / CONFIG_DRIVE_FOLDER / "lmstudio"

def copy_login_files(source, destination):
    copied = 0
    for directory in AUTH_DIRS:
        root = source / directory
        if not root.is_dir() or root.is_symlink():
            continue
        for original in root.rglob("*"):
            if not original.is_file() or original.is_symlink():
                continue
            relative = original.relative_to(source)
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, target)
            os.chmod(target, 0o600)
            copied += 1
    return copied

def restore_login():
    if not USE_GOOGLE_DRIVE:
        return False
    restored = copy_login_files(DRIVE_AUTH_DIR, LM_HOME)
    if restored:
        print(f"[INFO] GoogleドライブからLM Link認証情報を復元: {restored}ファイル")
    else:
        print("[INFO] 保存済みのLM Link認証情報なし。初回ログインが必要じゃ。")
    return bool(restored)

def backup_login():
    if not USE_GOOGLE_DRIVE:
        return
    saved = copy_login_files(LM_HOME, DRIVE_AUTH_DIR)
    if not saved:
        print("[WARN] LM Linkは有効じゃが、認証ファイルが見つからずバックアップできなかったのじゃ。")
        print("[WARN] Colabのランタイムを作り直した際、再ログインが必要になる可能性があるぞ。")
        for directory_name in ("credentials", "hub", ".internal"):
            root = LM_HOME / directory_name
            count = sum(1 for entry in root.rglob("*") if entry.is_file() and not entry.is_symlink()) if root.is_dir() else 0
            print(f"[INFO] 保存候補 {directory_name}: {count}個の通常ファイル")
        for directory in (LM_HOME, Path.home() / ".config" / "LM Studio", Path.home() / ".cache" / "lm-studio"):
            if directory.is_dir():
                names = sorted(item.name for item in directory.iterdir() if item.is_dir() and not item.is_symlink())
                print(f"[INFO] 認証保存先の調査: {directory} / サブディレクトリー: {names}")
        return False
    print(f"[INFO] LM Link認証情報をGoogleドライブに保存: {saved}ファイル")
    return True

auth_restored = restore_login()

## 5. LM Linkのログインと接続

保存済み認証情報が有効なら再ログインせずに接続する。初回またはログイン期限切れなら `lms login` の認証URLをブラウザで開くのじゃ。ログイン後に認証情報を保存する。

最初の「すべての設定」で `FORCE_RELOGIN` をオンにすると、一度ログアウトしてブラウザ認証をやり直せる。認証ファイルの保存先は調査中じゃ。

In [ ]:
#@title LM Linkログインと接続
import subprocess
import time

run_console(["lms", "daemon", "up"])
time.sleep(3)
whoami = run_console(["lms", "whoami"], check=False)
if FORCE_RELOGIN and whoami.returncode == 0:
    print("LM Studioから一度ログアウトし、再認証するのじゃ。", flush=True)
    run_console(["lms", "logout"])
if whoami.returncode != 0 or FORCE_RELOGIN:
    print("LM Studioへのログインが必要じゃ。表示される認証URLで承認するのじゃ。", flush=True)
    run_console(["lms", "login"])
else:
    print("現在のLM Studioログインは有効じゃ。再認証にはFORCE_RELOGINを有効にするのじゃ。")
run_console(["lms", "link", "enable"])
backup_login()
run_console(["lms", "link", "status"])
print("LM Linkの接続処理が完了したのじゃ。")


## 6. PLaMo 3 Translateモデルのダウンロードとロード

最初の「すべての設定」でモデルを選択する。既定は**8B／Q4_K_M**。0.8B・2B・8B・31Bからサイズを選び、pfnetが公式に配布しているGGUFの量子化形式だけ指定する。`lms get`の検索処理を使わず、PLaMo公式のGGUFファイルを直接取得するため、問題のコマンドは実行しない。ダウンロード中は進捗バー・転送量を表示し、中断時は可能なら途中から再開する。モデルの読込はLM Linkを一時停止してColab内のモデルだけを一覧化し、GGUFファイルのパスに対応する実際の`modelKey`を取得して実行する。読込後にLM Linkを再有効化する。Colabランタイムを破棄するとモデルファイルは消えるが、Googleドライブの保存済み認証情報は残る。31BはGPUメモリ不足でロードできない場合があるのじゃ。

Colabでロードする際の同時推論数は最初の設定欄で指定し、ロード後にローカルAPIが返す`config.parallel`もログに表示する。

In [ ]:
#@title PLaMoモデルの選択と読み込み
import os
import subprocess
from pathlib import Path

OFFICIAL_GGUF_FORMATS = {
    "0.8b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "2b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "8b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
    "31b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
}
if plamo_size not in OFFICIAL_GGUF_FORMATS:
    raise ValueError("未対応のPLaMoサイズじゃ")
if quantization not in OFFICIAL_GGUF_FORMATS[plamo_size]:
    raise ValueError(f"pfnet公式で配布されていない組み合わせじゃ: {plamo_size}/{quantization}")

repo_name = f"plamo-3-translate-{plamo_size}-GGUF"
filename = f"plamo-3-translate-{plamo_size}-{quantization}.gguf"
model_dir = Path.home() / ".lmstudio" / "models" / "pfnet" / repo_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / filename
partial_path = model_dir / (filename + ".part")
model_id = f"plamo-3-translate-{plamo_size}-colab"
url = f"https://huggingface.co/pfnet/{repo_name}/resolve/main/{filename}?download=true"

if not model_path.exists() or model_path.stat().st_size < 100_000_000:
    print(f"公式GGUFを取得: pfnet/{repo_name}/{filename}", flush=True)
    import re
    import time
    import requests
    from tqdm.auto import tqdm

    def download_with_progress(source_url, partial, complete):
        for attempt in range(1, 5):
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            try:
                with requests.get(source_url, headers=headers, stream=True, timeout=(30, 120)) as response:
                    if response.status_code == 416 and offset:
                        match = re.fullmatch(r"bytes \*/(\d+)", response.headers.get("Content-Range", ""))
                        if match and int(match.group(1)) == offset:
                            os.replace(partial, complete)
                            return
                    response.raise_for_status()
                    if response.status_code == 206:
                        match = re.fullmatch(
                            r"bytes (\d+)-(\d+)/(\d+|\*)",
                            response.headers.get("Content-Range", "")
                        )
                        if not match or int(match.group(1)) != offset:
                            raise RuntimeError("ダウンロードの再開位置が一致しないのじゃ")
                        total = int(match.group(3)) if match.group(3) != "*" else None
                        mode = "ab"
                    elif response.status_code == 200:
                        if offset:
                            print("サーバーが再開を受け付けなかったため、先頭から再取得するのじゃ。", flush=True)
                        offset = 0
                        total = int(response.headers["Content-Length"]) if response.headers.get("Content-Length") else None
                        mode = "wb"
                    else:
                        raise RuntimeError(f"予期しないダウンロード応答: HTTP {response.status_code}")
                    print(f"ダウンロード開始: {offset / 1048576:.1f} MiB取得済み", flush=True)
                    with partial.open(mode) as output, tqdm(
                        total=total, initial=offset, unit="B", unit_scale=True,
                        desc="PLaMo GGUF", mininterval=0.5, leave=True
                    ) as progress:
                        for chunk in response.iter_content(chunk_size=1048576):
                            if chunk:
                                output.write(chunk)
                                progress.update(len(chunk))
                    current = partial.stat().st_size
                    if total is not None and current != total:
                        raise RuntimeError(f"サイズ不一致: {current} / {total}バイト")
                    if current < 100_000_000:
                        raise RuntimeError("GGUFのダウンロード結果が小さすぎるのじゃ")
                    os.replace(partial, complete)
                    print("GGUFのダウンロードが完了したのじゃ。", flush=True)
                    return
            except (requests.RequestException, OSError) as error:
                if attempt == 4:
                    raise
                print(f"通信が中断したのじゃ（{attempt}/4）: {error}。途中から再試行するぞ。", flush=True)
                time.sleep(min(attempt * 2, 8))
        raise RuntimeError("GGUFのダウンロードに失敗したのじゃ")

    download_with_progress(url, partial_path, model_path)
else:
    print(f"ダウンロード済みGGUFを再利用: {filename}", flush=True)

import json
import re

def read_lms_json(arguments):
    result = run_console(arguments)
    raw = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", result.stdout).replace("\r", "")
    decoder = json.JSONDecoder()
    for index, character in enumerate(raw):
        if character not in "[{":
            continue
        try:
            parsed, _ = decoder.raw_decode(raw, index)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict) and isinstance(parsed.get("models"), list):
            return parsed["models"]
        if isinstance(parsed, dict) and isinstance(parsed.get("data"), list):
            return parsed["data"]
    raise RuntimeError(f"LM StudioのJSON一覧を解析できぬ: {arguments}")

expected_path = f"pfnet/{repo_name}/{filename}"
print("[INFO] LM Linkを一時停止し、Colabのローカルモデルだけを読み込むのじゃ。", flush=True)
run_console(["lms", "link", "disable"])
try:
    inventory = read_lms_json(["lms", "ls", "--json"])
    candidates = [
        item for item in inventory
        if isinstance(item, dict)
        and isinstance(item.get("path"), str)
        and item["path"].replace("\\", "/").lstrip("/").casefold() == expected_path.casefold()
        and isinstance(item.get("modelKey"), str)
        and item["modelKey"]
    ]
    if len(candidates) != 1:
        paths = [item.get("path") for item in inventory if isinstance(item, dict)]
        raise RuntimeError(
            f"ColabのローカルGGUFが一覧で一意に見つからぬ: {expected_path}。"
            f" 一覧のファイルパス: {paths}"
        )
    model_key = candidates[0]["modelKey"]
    print(f"[INFO] LM Studioが返したモデルキー: {model_key}", flush=True)
    print(f"[INFO] 対象GGUF: {expected_path}", flush=True)
    run_console(["lms", "unload", "--all"])
    run_console(["lms", "load", model_key, "--gpu", "max", "--parallel", str(PARALLEL_REQUESTS), "--identifier", model_id, "-y"])

    loaded = read_lms_json(["lms", "ps", "--json"])
    instances = [
        item for item in loaded
        if isinstance(item, dict) and item.get("identifier") == model_id
    ]
    if len(instances) != 1 or not isinstance(instances[0].get("path"), str) or (
        instances[0]["path"].replace("\\", "/").lstrip("/").casefold() != expected_path.casefold()
    ):
        print("[ERROR] 読み込まれたモデルのファイルが選択内容と一致しないのじゃ。", flush=True)
        run_console(["lms", "ps"])
        run_console(["lms", "unload", "--all"])
        raise RuntimeError(f"選択したGGUFとは異なるモデルがロードされたのじゃ: {expected_path}")
    print(f"[INFO] 指定したGGUFをColabでロードできたのじゃ: {model_id}", flush=True)
finally:
    print("[INFO] LM Linkの接続を復元するのじゃ。", flush=True)
    run_console(["lms", "link", "enable"])

run_console(["lms", "server", "start", "--port", "1234"])
run_console(["lms", "ps"])

import requests

try:
    info = requests.get("http://127.0.0.1:1234/api/v1/models", timeout=15)
    info.raise_for_status()
    instances = [
        instance
        for entry in info.json().get("models", [])
        for instance in entry.get("loaded_instances", [])
        if isinstance(instance, dict) and instance.get("id") == model_id
    ]
    if len(instances) != 1:
        print(f"[WARN] ローカルAPIで対象モデルの並列設定を確認できぬ: {model_id}", flush=True)
    else:
        actual = instances[0].get("config", {}).get("parallel")
        print(f"[INFO] Colabの同時推論設定: 指定={PARALLEL_REQUESTS}, API報告値={actual}", flush=True)
        if isinstance(actual, int) and actual != PARALLEL_REQUESTS:
            raise RuntimeError(
                f"並列数の設定が反映されておらぬ: 指定={PARALLEL_REQUESTS}、実際={actual}"
            )
        if actual is None:
            print("[WARN] APIがparallel値を返さぬ。実際のスロット数はエンジンログで確認するのじゃ。", flush=True)
except requests.RequestException as error:
    print(f"[WARN] モデルの並列設定のAPI確認に失敗: {error}", flush=True)

print(f"ロード完了: {model_id}（LM Linkから利用可能）", flush=True)
